In [ ]:
# import
import pandas as pd
import numpy as np
import warnings
import duckdb
import seaborn as sns

In [ ]:
# preferences
warnings.filterwarnings("ignore")

In [ ]:
# damage data
damage_df=pd.read_excel(open('damage_report_2022.xlsx', 'rb'), sheet_name='Material Receving Status Report', header=0, index_col=None, usecols='B, F, L')
qry='''
select ClaimDate claim_date, DistributorCode distrib_code, SKUName material_desc
from damage_df; 
'''
damage_df=duckdb.query(qry).df()
damage_df

In [ ]:
# PH data
ph_df=pd.read_excel(open('01. PH2022-10th Jan 23.xlsx', 'rb'), sheet_name='Selling code_Jan 2023', header=0, index_col=None) 
qry='''
select distinct "Pack size desc." pack_desc, "Material Description" material_desc
from ph_df;
'''
ph_df=duckdb.query(qry).df()
ph_df

In [ ]:
# COTC/Premium/MD data
cotc_df=pd.read_excel(open('IOP 2023 - COTC, MD & Premium List - Final.xlsx', 'rb'), sheet_name='COTC, MD & Premium List', header=0, index_col=None) 
qry='''
select distinct upper(SKUs) pack_desc, "Category" cat, "Business Group" bg
from cotc_df
where
    "COTC in 2023?" ilike '%yes%'
    -- or "Premium in 2023?" ilike '%yes%'
    -- or "MD in 2023?" ilike '%yes%';
'''
cotc_df=duckdb.query(qry).df()
cotc_df['brand']=cotc_df['pack_desc'].apply(lambda x: x.split()[0])
cotc_df

In [ ]:
# COTC damage data
qry='''
select left(claim_date::text, 7) claim_month, distrib_code, brand
from 
    damage_df tbl1 
    inner join 
    ph_df tbl2 using(material_desc)
    inner join 
    cotc_df tbl3 using(pack_desc);
'''
ret_df=duckdb.query(qry).df()
ret_df

In [ ]:
# brands
brands=set(ret_df['brand'])
print(brands)

In [ ]:
# cohort
sku_cohort_num_df=pd.DataFrame()
sku_cohort_pct_df=pd.DataFrame()

for b in brands: 
    qry='''
    with
        cohort as
        (select 
            tbl1.claim_month claim_month_from, 
            tbl2.claim_month claim_month_to, 
            datediff('month', (tbl1.claim_month || '-01')::date, (tbl2.claim_month || '-01')::date) month_diff, 
            count(distinct tbl1.distrib_code) distributors_ret
        from 
            (select claim_month, distrib_code, brand
            from ret_df
            where brand='''+"'"+b+"'"+'''
            ) tbl1 

            left join 

            (select claim_month, distrib_code, brand
            from ret_df
            where brand='''+"'"+b+"'"+'''
            ) tbl2 on(tbl1.distrib_code=tbl2.distrib_code and tbl1.claim_month<=tbl2.claim_month)
        group by 1, 2, 3
        ) 

    select *, distributors_ret*1.00/distributors_init distributors_ret_pct
    from 
        cohort tbl1

        inner join 

        (select claim_month_from, distributors_ret distributors_init
        from cohort
        where month_diff=0
        ) tbl2 using(claim_month_from)
    order by 1, 2; 
    '''
    cohort_df=duckdb.query(qry).df()

    # cohort (number)
    cohort_df_piv=pd.pivot_table(cohort_df, index='claim_month_from', columns='month_diff', values='distributors_ret', aggfunc='sum')
    cohort_df_piv.loc['avg.']=cohort_df_piv.mean(axis=0)
    cohort_df_piv['brand']=b
    sku_cohort_num_df=sku_cohort_num_df._append(cohort_df_piv.tail(1))
    
    # cohort (pct)
    cohort_df_piv=pd.pivot_table(cohort_df, index='claim_month_from', columns='month_diff', values='distributors_ret_pct', aggfunc='sum')
    cohort_df_piv.loc['avg.']=cohort_df_piv.mean(axis=0)
    cohort_df_piv['brand']=b
    sku_cohort_pct_df=sku_cohort_pct_df._append(cohort_df_piv.tail(1))
    
    print("Cohorts generated for brand: "+b)

display(sku_cohort_num_df)
display(sku_cohort_pct_df)

In [ ]:
# style
cm=sns.color_palette("blend:white,blue", as_cmap=True)
cm.set_bad("white")

In [ ]:
# show pct cohort
df=sku_cohort_pct_df
cols=list(df.columns)
cols=[cols[-1]]+cols[:-1]
df=df[cols].reset_index(drop=True)
df.style.format(formatter="{:.2%}", subset=df.select_dtypes(include=np.number).columns.tolist()).background_gradient(cmap=cm, axis=None, low=0.1, high=1.0).hide()

In [ ]:
# show num cohort
df=sku_cohort_num_df
cols=list(df.columns)
cols=[cols[-1]]+cols[:-1]
df=df[cols].reset_index(drop=True)
df.style.format(formatter="{:.2f}", subset=df.select_dtypes(include=np.number).columns.tolist()).background_gradient(cmap=cm, axis=None, low=0.1, high=1.0).hide()